# PyTorch mechanics

MichAl Academy, unit 3.0.

Run each cell with **Shift+Enter**.

Every notebook from here on runs on PyTorch. This one is the tour of the six
pieces the rest of the track assumes you have seen: tensors, autograd,
`nn.Module`, `DataLoader`, the training loop, and devices.

Nothing here is a network yet. It is the machinery a network is built out of.


In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_digits
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())


## 3.0.1 Tensors

Three attributes describe any tensor: its shape, the kind of number it holds,
and where it is stored.


In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("shape ", a.shape)
print("dtype ", a.dtype)
print("device", a.device)
print("ndim  ", a.dim(), " numel", a.numel())


The dtype trap. numpy defaults to 64-bit floats and PyTorch to 32-bit, so a
tensor built from a numpy array keeps the wider type and a layer refuses it.


In [ ]:
arr = np.array([1.0, 2.0, 3.0])
from_numpy = torch.from_numpy(arr)

print("numpy dtype        :", arr.dtype)
print("torch.from_numpy   :", from_numpy.dtype)
print("torch.tensor([1.0]):", torch.tensor([1.0]).dtype)

layer = nn.Linear(3, 2)
try:
    layer(from_numpy.unsqueeze(0))
except RuntimeError as e:
    print("\nfloat64 into a float32 layer ->", e)


`torch.from_numpy` also shares memory with the array rather than copying it.


In [ ]:
arr[0] = 99.0
print("the tensor changed too:", from_numpy.tolist())

# Say the dtype when you build the tensor and neither problem happens.
safe = torch.tensor(arr, dtype=torch.float32)
print("explicit dtype      :", safe.dtype)


## 3.0.2 Autograd

Mark a tensor with `requires_grad=True`, do arithmetic, call `.backward()` on
the result. PyTorch fills in the derivative with respect to every marked tensor.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()

print("y      =", y.item())
print("x.grad =", x.grad.item(), " (the derivative of x**2 is 2x, and 2*3 = 6)")


Two parameters, so the chain has something to divide between.

s = w*4 + b = 9, so out = 81. The derivative of s squared is 2s = 18. w enters
multiplied by 4, so its gradient is 18 * 4; b is added directly, so its
gradient is 18 * 1.


In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

out = (w * 4 + b) ** 2
out.backward()

print("out    =", out.item())
print("w.grad =", w.grad.item(), "expected", 18 * 4)
print("b.grad =", b.grad.item(), "expected", 18 * 1)


Gradients **add** to what is already there. This is the reason every training
loop calls `zero_grad()`.


In [ ]:
z = torch.tensor(3.0, requires_grad=True)
for i in range(3):
    (z ** 2).backward()
    print(f"backward #{i + 1}: z.grad = {z.grad.item()}")


## 3.0.3 nn.Module

A layer assigned to an attribute is registered, so `parameters()` returns
everything training is allowed to change.


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(64, 32)
        self.out = nn.Linear(32, 10)

    def forward(self, x):
        return self.out(torch.relu(self.hidden(x)))


net = Net()
for name, p in net.named_parameters():
    print(f"{name:14s} {str(tuple(p.shape)):10s} {p.numel():5d}")

total = sum(p.numel() for p in net.parameters())
print(f"\n{len(list(net.parameters()))} tensors, {total} parameters")
print(f"the widest weight matrix is {2048 / total:.0%} of the model")


Layers in a plain Python list are invisible to `parameters()`. The forward pass
still works, so the error arrives one line later and does not mention the list.


In [ ]:
class BadNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(64, 32), nn.Linear(32, 10)]

    def forward(self, x):
        return self.layers[1](torch.relu(self.layers[0](x)))


bad = BadNet()
print("parameters() returns:", len(list(bad.parameters())), "tensors")

try:
    torch.optim.SGD(bad.parameters(), lr=0.1)
except ValueError as e:
    print("optimiser ->", e)

# nn.Sequential and nn.ModuleList both register properly.
seq = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
print("\nSequential:", len(list(seq.parameters())), "tensors,",
      sum(p.numel() for p in seq.parameters()), "parameters")


## 3.0.4 Dataset and DataLoader

`load_digits` is 1797 handwritten digits, each an 8 by 8 grid of brightness
values, already flattened to a row of 64. Dividing by 16 puts every value
between 0 and 1.


In [ ]:
digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32) / 16.0
Y = torch.tensor(digits.target, dtype=torch.long)
print("X", tuple(X.shape), X.dtype)
print("Y", tuple(Y.shape), Y.dtype)

ds = TensorDataset(X, Y)
dl = DataLoader(ds, batch_size=64, shuffle=True)

sizes = [len(yb) for _, yb in dl]
print(f"\n{len(sizes)} batches, the last holding {sizes[-1]}")
print(f"28 * 64 = {28 * 64}, leaving {len(Y) - 28 * 64}")

for bs in (1, 32, 64, 256, len(Y)):
    print(f"batch_size {bs:5d} -> {len(DataLoader(ds, batch_size=bs)):5d} batches")


Why shuffling is not optional. Sort the data by label first, then look at what
the first batch contains.


In [ ]:
order = torch.argsort(Y)
sorted_ds = TensorDataset(X[order], Y[order])

for shuffle in (False, True):
    first = next(iter(DataLoader(sorted_ds, batch_size=64, shuffle=shuffle)))
    print(f"shuffle={str(shuffle):5s} -> {len(set(first[1].tolist()))} classes in the first batch")


## 3.0.5 The training loop

Five lines, in this order. Everything else in a real script is logging,
checkpoints and evaluation built around them.


In [ ]:
def train(zero_grad=True, epochs=20, seed=0, lr=0.1):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X, Y), batch_size=64, shuffle=True)

    per_epoch = []
    for _ in range(epochs):
        running = 0.0
        for xb, yb in loader:
            pred = model(xb)             # 1. forward
            loss = loss_fn(pred, yb)     # 2. how wrong
            if zero_grad:
                opt.zero_grad()          # 3. clear last step's gradients
            loss.backward()              # 4. fill .grad for every parameter
            opt.step()                   # 5. parameter -= lr * gradient
            running += loss.item() * len(yb)
        per_epoch.append(running / len(Y))

    with torch.no_grad():
        acc = (model(X).argmax(1) == Y).float().mean().item()
    return per_epoch, acc


good, acc_good = train(zero_grad=True)
print("epoch 1 loss :", round(good[0], 3), " (an untrained 10-class model scores ln(10) =",
      round(float(np.log(10)), 3), ")")
print("epoch 20 loss:", round(good[-1], 3))
print("training accuracy:", round(acc_good, 3))


Now delete one line. Gradients accumulate, so each step carries every step
before it.


In [ ]:
bad_run, acc_bad = train(zero_grad=False)

print("epoch | with zero_grad | without")
for i in (0, 4, 8, 12, 16, 19):
    print(f"{i + 1:5d} | {good[i]:14.2f} | {bad_run[i]:7.2f}")
print(f"\naccuracy: {acc_good:.3f} with, {acc_bad:.3f} without")


The first epoch of the broken run has the lower loss of the two, because an
accumulated gradient is a bigger step and early on a bigger step helps. It does
not last.

Here is the accumulation itself, measured as the length of the gradient vector
over all 2,410 parameters, for the first five updates each way.


In [ ]:
def gradient_lengths(zero_grad, steps=5, seed=0, lr=0.1):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X, Y), batch_size=64, shuffle=True)

    lengths = []
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        if zero_grad:
            opt.zero_grad()
        loss.backward()
        flat = torch.cat([p.grad.flatten() for p in model.parameters()])
        lengths.append(flat.norm().item())
        opt.step()
        if len(lengths) == steps:
            return lengths


print("step            ", "  ".join(f"{i + 1:5d}" for i in range(5)))
print("with zero_grad  ", "  ".join(f"{v:5.2f}" for v in gradient_lengths(True)))
print("without         ", "  ".join(f"{v:5.2f}" for v in gradient_lengths(False)))


## 3.0.6 Devices

Every tensor is on the CPU or on one particular graphics card, and an operation
needs all of its tensors on the same one.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
model.to(device)                     # moves the model in place
xb = X[:64].to(device)               # returns a copy, so the assignment matters
print("model parameter is on:", next(model.parameters()).device)
print("batch is on           :", xb.device)


The asymmetry, shown with dtype rather than device so it runs on any machine:
`.to()` on a model changes it, `.to()` on a tensor hands back a copy.


In [ ]:
m = nn.Linear(3, 2)
m.to(torch.float64)
print("model after m.to(float64) :", m.weight.dtype)

t = torch.zeros(3)
t.to(torch.float64)
print("tensor after t.to(float64):", t.dtype, "  <- unchanged, the copy was thrown away")
print("the returned copy         :", t.to(torch.float64).dtype)


This course does not need a GPU. Time the whole of the training run above.


In [ ]:
import time

start = time.perf_counter()
train(epochs=20)
print(f"20 epochs of the 64-32-10 network: {time.perf_counter() - start:.1f} s on {device}")


## What to try

1. Change `batch_size` to 8 and to 512 and rerun the training cell. Count the
   weight updates in each case before you look at the loss.
2. Set `lr=1.0` and then `lr=0.001`. One diverges and one barely moves; unit 3.5
   is about where the edge sits.
3. Build `BadNet` with `nn.ModuleList` instead of a plain list and confirm
   `parameters()` returns 4 tensors again.
